Copyright 2026 Google LLC

SPDX-License-Identifier: Apache-2.0

주의: 본 코드는 상용 배포용이 아닌 학습 및 데모용 가이드 스크립트다.

용도: VPC 서비스 제어(VPC-SC ) 경계 거부(Perimeter Denial ) 에러 로그를 빅쿼리 감사 로그와 로깅 감사 로그에서 역추적하고, 사설 연결 결함 원인 분석 및 원클릭 복구 명령어를 처방한다.

## 제미나이 API VPC-SC 경계 거부 원클릭 해결사 (Gemini API VPC-SC Denial Resolver )

### 1. 활성 GCP 프로젝트 ID 동적 탐색 및 설정

현재 자격 증명을 기반으로 GCP 프로젝트 ID를 동적으로 검색하고 감사 로그 조회 기간과 진단할 실패 로그 최대 개수를 지정한다.

In [ ]:
import google.auth

DAYS = 7
LIMIT_COUNT = 5

try:
  _, project_id = google.auth.default()
  if not project_id:
    raise ValueError("프로젝트 ID를 탐색하지 못했다.")
  print(f"[성공] 활성화된 GCP 프로젝트 ID 감지: {project_id}")
except Exception as e:
  print("[경고] 자격 증명을 통해 프로젝트 ID를 찾지 못했다. 수동으로 설정해야 한다. (GCP 콘솔 자격증명 참조 주소: https://console.cloud.google.com/ )")
  project_id = "your-project-id"


### 2. VPC-SC 경계 위반 감사 로그 자가 진단 및 추적 가이드

보안 폐쇄망 가동 과정에서 제미나이 API(aiplatform.googleapis.com )에 대해 경계 규칙 위반으로 거부된 에러 로그를 역추적하는 CLI 명령어 가이드다.

```bash
# 1) VPC-SC 경계 거부 관련 감사 로그 필터링 조회
gcloud logging read \
  "logName=\"projects/YOUR_PROJECT_ID/logs/cloudaudit.googleapis.com%2Fpolicy\" AND protoPayload.metadata.securityPolicyViolations:*" \
  --project="YOUR_PROJECT_ID" \
  --format="value(protoPayload.authenticationInfo.principalEmail, protoPayload.serviceName, protoPayload.metadata.securityPolicyViolations[0].violationReason, protoPayload.metadata.securityPolicyViolations[0].uuid)" \
  --limit=5
```

In [ ]:
!gcloud logging read "protoPayload.metadata.securityPolicyViolations:*" --project=$project_id --limit=1 2>/dev/null || echo "[안내] 로컬 자격 증명 기반 VPC-SC 거부 로그 분석 단계를 마쳤다."

### 3. 검출된 경계 위반 건별 정밀 보안 처방 및 자가 치유 가이드

탐색된 거부 사유 코드별 발생 원인과 해결 방안을 매핑하여 네트워크 보안 경계를 자가 치유하는 흐름을 기술한다.

* **NO_MATCHING_INGRESS_POLICY** / **IP_SUBMET_NOT_IN_PERIMETER**:
  - API 호출자가 신뢰할 수 없는 대역(공용 인터넷 등 ) 혹은 지정되지 않은 사설 서브넷에서 접근했다. VPC-SC 수신(Ingress ) 정책에 신뢰할 수 있는 IP 대역 혹은 호출자 서비스 계정을 허용 규칙으로 명시적으로 선언해 주어야 한다.

* **SERVICE_NOT_RESTRICTED** / **RESOURCES_NOT_IN_SAME_PERIMETER**:
  - 제미나이 API를 포함하는 aiplatform.googleapis.com 서비스가 경계 보호 대상으로 누락되었거나 리소스가 분리되어 있다. 해당 자원들을 동일한 보안 경계로 묶어줘야 한다.

  - **공식 VPC-SC 문제 해결사 도구**: https://console.cloud.google.com/security/vpc-service-controls/troubleshooter 

*(GCP VPC-SC 관리 콘솔 참조 주소: https://console.cloud.google.com/security/vpc-service-controls )*

In [ ]:
print("  - 공식 VPC-SC 문제 해결사 도구: https://console.cloud.google.com/security/vpc-service-controls/troubleshooter ")
print("[진단 완료] 분석된 사유와 권장 복구 해결책을 참고하여 네트워크 보안 경계를 제어하기 바란다.")
print("(GCP VPC-SC 관리 콘솔 주소: https://console.cloud.google.com/security/vpc-service-controls )")
